# DPF CH-SIMS

In [11]:
import pickle
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import cs16.DPF as DPF

# ============================================================
# 1. CH-SIMS
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'

with open(file_path, 'rb') as f:
    data = pickle.load(f)

print("CH-SIMS Data keys:", data.keys())

train = data['train']
valid = data['valid']
test = data['test']

print("\nTrain keys:", train.keys())

# ============================================================
# 2. Retrieve
# ============================================================
X_text_train = train['text']
X_vision_train = train['vision']

# Convert to int
y_train = train['classification_labels'].astype(int)
y_val = valid['classification_labels'].astype(int)
y_test = test['classification_labels'].astype(int)

X_text_val = valid['text']
X_vision_val = valid['vision']

X_text_test = test['text']
X_vision_test = test['vision']

print(f"\nData shapes:")
print(f"  Text train: {X_text_train.shape}, Vision train: {X_vision_train.shape}")
print(f"  Text val:   {X_text_val.shape}, Vision val:   {X_vision_val.shape}")
print(f"  Text test:  {X_text_test.shape}, Vision test:  {X_vision_test.shape}")

# 现在 bincount 可以正常工作
print(f"\nLabel distribution (train): {np.bincount(y_train)}")
print(f"Label distribution (val): {np.bincount(y_val)}")
print(f"Label distribution (test): {np.bincount(y_test)}")


def aggregate_sequence(X, method='mean'):
    if method == 'mean':
        return np.mean(X, axis=1)
    elif method == 'max':
        return np.max(X, axis=1)
    elif method == 'concat':
        mean = np.mean(X, axis=1)
        std = np.std(X, axis=1)
        return np.concatenate([mean, std], axis=1)
    else:
        return X

X_text_train = aggregate_sequence(X_text_train, method='concat')
X_vision_train = aggregate_sequence(X_vision_train, method='concat')
X_text_val = aggregate_sequence(X_text_val, method='concat')
X_vision_val = aggregate_sequence(X_vision_val, method='concat')
X_text_test = aggregate_sequence(X_text_test, method='concat')
X_vision_test = aggregate_sequence(X_vision_test, method='concat')

CH-SIMS Data keys: dict_keys(['train', 'valid', 'test'])

Train keys: dict_keys(['raw_text', 'text_bert', 'audio_lengths', 'vision_lengths', 'classification_labels', 'regression_labels', 'classification_labels_T', 'regression_labels_T', 'classification_labels_A', 'regression_labels_A', 'classification_labels_V', 'regression_labels_V', 'text', 'audio', 'vision', 'id'])

Data shapes:
  Text train: (1368, 39, 768), Vision train: (1368, 55, 709)
  Text val:   (456, 39, 768), Vision val:   (456, 55, 709)
  Text test:  (457, 39, 768), Vision test:  (457, 55, 709)

Label distribution (train): [742 207 419]
Label distribution (val): [248  69 139]
Label distribution (test): [248  69 140]


In [61]:
"""
CH-SIMS: ExtraTree Unimodal + DPF + XGBoost Meta-Classifier (Binary Classification)

This script:
- Loads CH-SIMS data from SIMS/Processed/unaligned_39.pkl
- Converts to binary classification (removes neutral samples)
- Trains ExtraTreesClassifier as unimodal classifier for text and vision modalities
- Applies DPF fusion with fixed beta
- Evaluates using XGBoost as meta-classifier
- Reports accuracy, F1, and runtime
"""

import pickle
import time
import numpy as np
import xgboost as xgb
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import cs16.DPF as DPF
start_time = time.time()
# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

# ============================================================
# 2. Feature Aggregation (Mean over time dimension)
# ============================================================
def aggregate_mean(X):
    """Aggregate features by taking mean over time dimension."""
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

print("=" * 60)
print("CH-SIMS Data Shapes (Original 3-class)")
print("=" * 60)
print(f"Train: Text {X_text_train.shape}, Vision {X_vision_train.shape}")
print(f"Val:   Text {X_text_val.shape}, Vision {X_vision_val.shape}")
print(f"Test:  Text {X_text_test.shape}, Vision {X_vision_test.shape}")
print(f"Labels (train): {np.bincount(y_train)}")
print(f"Labels (val): {np.bincount(y_val)}")
print(f"Labels (test): {np.bincount(y_test)}")
print("=" * 60)

# ============================================================
# 3. Convert to Binary Classification (Remove Neutral)
# ============================================================
print("\nConverting to binary classification (removing neutral samples)...")

def convert_to_binary(X_text, X_vision, y):
    """Remove neutral samples (class 1) and remap labels: 0->0, 2->1"""
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

print(f"Binary classification shapes:")
print(f"Train: Text {X_text_train_bin.shape}, Vision {X_vision_train_bin.shape}, Labels {np.bincount(y_train_bin)}")
print(f"Val:   Text {X_text_val_bin.shape}, Vision {X_vision_val_bin.shape}, Labels {np.bincount(y_val_bin)}")
print(f"Test:  Text {X_text_test_bin.shape}, Vision {X_vision_test_bin.shape}, Labels {np.bincount(y_test_bin)}")
print("=" * 60)

start_time = time.time()

# ============================================================
# 4. Text Modality: ExtraTreesClassifier Unimodal
# ============================================================
print("\nTraining ExtraTree Text Unimodal (Binary)...")

clf_text = ExtraTreesClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

clf_text.fit(X_text_train_bin, y_train_bin)

train_probs_text = clf_text.predict_proba(X_text_train_bin)
val_probs_text = clf_text.predict_proba(X_text_val_bin)
test_probs_text = clf_text.predict_proba(X_text_test_bin)

print(f"Text probs shape: Train {train_probs_text.shape}, Test {test_probs_text.shape}")

# ============================================================
# 5. Vision Modality: ExtraTreesClassifier Unimodal
# ============================================================
print("Training ExtraTree Vision Unimodal (Binary)...")

clf_vision = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=6,
    max_features='sqrt',
    class_weight='balanced',
    bootstrap=True,
    random_state=42
)

clf_vision.fit(X_vision_train_bin, y_train_bin)

train_probs_vision = clf_vision.predict_proba(X_vision_train_bin)
val_probs_vision = clf_vision.predict_proba(X_vision_val_bin)
test_probs_vision = clf_vision.predict_proba(X_vision_test_bin)

print(f"Vision probs shape: Train {train_probs_vision.shape}, Test {test_probs_vision.shape}")

# ============================================================
# 6. DPF Fusion
# ============================================================
BETA = 1.1
TOPK = 2
DELTA = 1e-2

print(f"\nApplying DPF Fusion (β={BETA}, topk={TOPK}, delta={DELTA})...")

X_train_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    train_probs_text, train_probs_vision, beta=BETA, topk=TOPK, delta=DELTA
)
X_val_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    val_probs_text, val_probs_vision, beta=BETA, topk=TOPK, delta=DELTA
)
X_test_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    test_probs_text, test_probs_vision, beta=BETA, topk=TOPK, delta=DELTA
)

print(f"Dynamic Fusion Feature shape (train): {X_train_global.shape}")

# ============================================================
# 7. Downstream: XGBoost Meta-Classifier
# ============================================================
print("\nTraining XGBoost Meta-Classifier (Binary)...")

global_clf = xgb.XGBClassifier(
    n_estimators=350,
    max_depth=5,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    verbosity=0
)

global_clf.fit(X_train_global, y_train_bin)

# ============================================================
# 8. Evaluation (Weighted F1)
# ============================================================
test_pred = global_clf.predict(X_test_global)

print("\n" + "=" * 60)
print("CH-SIMS Results with ExtraTree + DPF (Binary Classification)")
print("=" * 60)
print(f"Test Accuracy: {accuracy_score(y_test_bin, test_pred):.4f}")
print(f"Test Weighted F1: {f1_score(y_test_bin, test_pred, average='weighted'):.4f}")
print(f"Test Binary F1: {f1_score(y_test_bin, test_pred, average='binary'):.4f}")
print("\nTest Classification Report:")
print(classification_report(
    y_test_bin, test_pred,
    digits=4,
    target_names=['negative', 'positive']
))

# ============================================================
# 9. Uniform Baseline (β=0)
# ============================================================
print("\n" + "=" * 60)
print("Uniform Baseline (β=0) - Binary")
print("=" * 60)

X_train_uniform = DPF.enhanced_dynamic_fusion_topk_adaptive(
    train_probs_text, train_probs_vision, beta=0.0, topk=TOPK, delta=DELTA
)
X_test_uniform = DPF.enhanced_dynamic_fusion_topk_adaptive(
    test_probs_text, test_probs_vision, beta=0.0, topk=TOPK, delta=DELTA
)

clf_uniform = xgb.XGBClassifier(
    n_estimators=350,
    max_depth=5,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    verbosity=0
)

clf_uniform.fit(X_train_uniform, y_train_bin)
uniform_pred = clf_uniform.predict(X_test_uniform)

print(f"Uniform Baseline Test Accuracy: {accuracy_score(y_test_bin, uniform_pred):.4f}")
print(f"Uniform Baseline Test Weighted F1: {f1_score(y_test_bin, uniform_pred, average='weighted'):.4f}")

# ============================================================
# 10. Unimodal Baselines
# ============================================================
print("\n" + "=" * 60)
print("Unimodal Baselines (Binary)")
print("=" * 60)

text_pred = clf_text.predict(X_text_test_bin)
vision_pred = clf_vision.predict(X_vision_test_bin)

print(f"Text-only Test Accuracy: {accuracy_score(y_test_bin, text_pred):.4f}")
print(f"Text-only Test Weighted F1: {f1_score(y_test_bin, text_pred, average='weighted'):.4f}")
print(f"Vision-only Test Accuracy: {accuracy_score(y_test_bin, vision_pred):.4f}")
print(f"Vision-only Test Weighted F1: {f1_score(y_test_bin, vision_pred, average='weighted'):.4f}")

# ============================================================
# 11. Summary Table
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS SUMMARY TABLE (Binary Classification)")
print("=" * 60)
print(f"{'Method':<25} {'Accuracy':<12} {'Weighted F1':<12}")
print("-" * 50)
print(f"{'DPF (Ours, β=1.1)':<25} {accuracy_score(y_test_bin, test_pred):.4f}      {f1_score(y_test_bin, test_pred, average='weighted'):.4f}")
print(f"{'Uniform (β=0)':<25} {accuracy_score(y_test_bin, uniform_pred):.4f}      {f1_score(y_test_bin, uniform_pred, average='weighted'):.4f}")
print(f"{'Text-only (ExtraTree)':<25} {accuracy_score(y_test_bin, text_pred):.4f}      {f1_score(y_test_bin, text_pred, average='weighted'):.4f}")
print(f"{'Vision-only (ExtraTree)':<25} {accuracy_score(y_test_bin, vision_pred):.4f}      {f1_score(y_test_bin, vision_pred, average='weighted'):.4f}")
print("=" * 60)

# ============================================================
# 12. Runtime
# ============================================================
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Execution Time: {elapsed_time:.4f} seconds")

CH-SIMS Data Shapes (Original 3-class)
Train: Text (1368, 768), Vision (1368, 709)
Val:   Text (456, 768), Vision (456, 709)
Test:  Text (457, 768), Vision (457, 709)
Labels (train): [742 207 419]
Labels (val): [248  69 139]
Labels (test): [248  69 140]

Converting to binary classification (removing neutral samples)...
Binary classification shapes:
Train: Text (1161, 768), Vision (1161, 709), Labels [742 419]
Val:   Text (387, 768), Vision (387, 709), Labels [248 139]
Test:  Text (388, 768), Vision (388, 709), Labels [248 140]

Training ExtraTree Text Unimodal (Binary)...
Text probs shape: Train (1161, 2), Test (388, 2)
Training ExtraTree Vision Unimodal (Binary)...
Vision probs shape: Train (1161, 2), Test (388, 2)

Applying DPF Fusion (β=1.1, topk=2, delta=0.01)...
Dynamic Fusion Feature shape (train): (1161, 4)

Training XGBoost Meta-Classifier (Binary)...

CH-SIMS Results with ExtraTree + DPF (Binary Classification)
Test Accuracy: 0.7912
Test Weighted F1: 0.7884
Test Binary F1: 0.6

C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


Uniform Baseline Test Weighted F1: 0.7403

Unimodal Baselines (Binary)
Text-only Test Accuracy: 0.7655
Text-only Test Weighted F1: 0.7403
Vision-only Test Accuracy: 0.7139
Vision-only Test Weighted F1: 0.7094

CH-SIMS SUMMARY TABLE (Binary Classification)
Method                    Accuracy     Weighted F1 
--------------------------------------------------
DPF (Ours, β=1.1)         0.7912      0.7884
Uniform (β=0)             0.7655      0.7403
Text-only (ExtraTree)     0.7655      0.7403
Vision-only (ExtraTree)   0.7139      0.7094

Total Execution Time: 1.0233 seconds


In [62]:
"""
CH-SIMS: ExtraTree Unimodal + DPF + XGBoost Meta-Classifier (Binary Classification)
Paper-Ready Version - Uses Weighted F1
"""

import pickle
import time
import numpy as np
import xgboost as xgb
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import cs16.DPF as DPF

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

# ============================================================
# 2. Convert to Binary Classification (Remove Neutral)
# ============================================================
def convert_to_binary(X_text, X_vision, y):
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

print("=" * 60)
print("CH-SIMS Binary Classification")
print("=" * 60)
print(f"Train: Text {X_text_train_bin.shape}, Vision {X_vision_train_bin.shape}")
print(f"Labels (train): {np.bincount(y_train_bin)}")
print(f"Labels (test): {np.bincount(y_test_bin)}")
print("=" * 60)

# ============================================================
# 3. Text Modality: ExtraTreesClassifier
# ============================================================
print("\nTraining ExtraTree Text (Binary)...")

clf_text = ExtraTreesClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
clf_text.fit(X_text_train_bin, y_train_bin)

train_probs_text = clf_text.predict_proba(X_text_train_bin)
val_probs_text = clf_text.predict_proba(X_text_val_bin)
test_probs_text = clf_text.predict_proba(X_text_test_bin)

# ============================================================
# 4. Vision Modality: ExtraTreesClassifier
# ============================================================
print("Training ExtraTree Vision (Binary)...")

clf_vision = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=6,
    max_features='sqrt',
    class_weight='balanced',
    bootstrap=True,
    random_state=42
)
clf_vision.fit(X_vision_train_bin, y_train_bin)

train_probs_vision = clf_vision.predict_proba(X_vision_train_bin)
val_probs_vision = clf_vision.predict_proba(X_vision_val_bin)
test_probs_vision = clf_vision.predict_proba(X_vision_test_bin)

# ============================================================
# 5. DPF Fusion (β=1.1)
# ============================================================
BETA = 1.1
TOPK = 2
DELTA = 1e-2

print(f"\nApplying DPF Fusion (β={BETA})...")

X_train_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    train_probs_text, train_probs_vision, beta=BETA, topk=TOPK, delta=DELTA
)
X_val_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    val_probs_text, val_probs_vision, beta=BETA, topk=TOPK, delta=DELTA
)
X_test_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    test_probs_text, test_probs_vision, beta=BETA, topk=TOPK, delta=DELTA
)

# ============================================================
# 6. XGBoost Meta-Classifier
# ============================================================
print("\nTraining XGBoost Meta-Classifier...")

global_clf = xgb.XGBClassifier(
    n_estimators=350,
    max_depth=5,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    verbosity=0
)
global_clf.fit(X_train_global, y_train_bin)

# ============================================================
# 7. Evaluation (Weighted F1)
# ============================================================
test_pred = global_clf.predict(X_test_global)

print("\n" + "=" * 60)
print("CH-SIMS RESULTS")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test_bin, test_pred):.4f}")
print(f"Weighted F1: {f1_score(y_test_bin, test_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(
    y_test_bin, test_pred, digits=4,
    target_names=['negative', 'positive']
))

# ============================================================
# 8. Baselines
# ============================================================
print("\n" + "=" * 60)
print("BASELINES")
print("=" * 60)

# Uniform (β=0)
X_train_uniform = DPF.enhanced_dynamic_fusion_topk_adaptive(
    train_probs_text, train_probs_vision, beta=0.0, topk=TOPK, delta=DELTA
)
X_test_uniform = DPF.enhanced_dynamic_fusion_topk_adaptive(
    test_probs_text, test_probs_vision, beta=0.0, topk=TOPK, delta=DELTA
)

clf_uniform = xgb.XGBClassifier(
    n_estimators=350, max_depth=5, random_state=42,
    eval_metric='logloss', use_label_encoder=False, verbosity=0
)
clf_uniform.fit(X_train_uniform, y_train_bin)
uniform_pred = clf_uniform.predict(X_test_uniform)

# Unimodal
text_pred = clf_text.predict(X_text_test_bin)
vision_pred = clf_vision.predict(X_vision_test_bin)

# Summary Table
print(f"\n{'Method':<25} {'Accuracy':<12} {'Weighted F1':<12}")
print("-" * 50)
print(f"{'DPF (β=1.1)':<25} {accuracy_score(y_test_bin, test_pred):.4f}      {f1_score(y_test_bin, test_pred, average='weighted'):.4f}")
print(f"{'Uniform (β=0)':<25} {accuracy_score(y_test_bin, uniform_pred):.4f}      {f1_score(y_test_bin, uniform_pred, average='weighted'):.4f}")
print(f"{'Text-only':<25} {accuracy_score(y_test_bin, text_pred):.4f}      {f1_score(y_test_bin, text_pred, average='weighted'):.4f}")
print(f"{'Vision-only':<25} {accuracy_score(y_test_bin, vision_pred):.4f}      {f1_score(y_test_bin, vision_pred, average='weighted'):.4f}")
print("=" * 60)

print(f"\nTotal Time: {time.time() - start_time:.4f} seconds")

CH-SIMS Binary Classification
Train: Text (1161, 768), Vision (1161, 709)
Labels (train): [742 419]
Labels (test): [248 140]

Training ExtraTree Text (Binary)...
Training ExtraTree Vision (Binary)...

Applying DPF Fusion (β=1.1)...

Training XGBoost Meta-Classifier...

CH-SIMS RESULTS
Accuracy:  0.7912
Weighted F1: 0.7884

Classification Report:
              precision    recall  f1-score   support

    negative     0.8175    0.8669    0.8415       248
    positive     0.7360    0.6571    0.6943       140

    accuracy                         0.7912       388
   macro avg     0.7767    0.7620    0.7679       388
weighted avg     0.7881    0.7912    0.7884       388


BASELINES

Method                    Accuracy     Weighted F1 
--------------------------------------------------
DPF (β=1.1)               0.7912      0.7884
Uniform (β=0)             0.7655      0.7403
Text-only                 0.7655      0.7403
Vision-only               0.7139      0.7094

Total Time: 2.1506 seconds


C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


# delta =0 （No Regularzation）

In [60]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import cs16.DPF as DPF

# ============================================================
# DPF Fusion
# ============================================================
beta = 1.1
topk = 2
delta =0

X_train_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    train_probs_text, train_probs_vision, beta, topk, delta
)
X_test_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    test_probs_text, test_probs_vision, beta, topk, delta
)

print("X_train_global shape:", X_train_global.shape)

# ============================================================
# Train Global Classifier (Binary)
# ============================================================
global_clf = xgb.XGBClassifier(
    n_estimators=350,
    max_depth=5,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    verbosity=0
)
global_clf.fit(X_train_global, y_train_bin)

# ============================================================
# Evaluation
# ============================================================
test_pred = global_clf.predict(X_test_global)

print("\nTest Accuracy:", accuracy_score(y_test_bin, test_pred))
print("\nTest Classification Report:")
print(classification_report(
    y_test_bin, test_pred, digits=4,
    target_names=['negative', 'positive']
))

X_train_global shape: (1161, 4)

Test Accuracy: 0.7912371134020618

Test Classification Report:
              precision    recall  f1-score   support

    negative     0.8175    0.8669    0.8415       248
    positive     0.7360    0.6571    0.6943       140

    accuracy                         0.7912       388
   macro avg     0.7767    0.7620    0.7679       388
weighted avg     0.7881    0.7912    0.7884       388



C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


# Beta 

In [59]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import cs16.DPF as DPF

# ============================================================
# DPF Fusion
# ============================================================
beta = 0
topk = 2
delta =1e-2

X_train_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    train_probs_text, train_probs_vision, beta, topk, delta
)
X_test_global = DPF.enhanced_dynamic_fusion_topk_adaptive(
    test_probs_text, test_probs_vision, beta, topk, delta
)

print("X_train_global shape:", X_train_global.shape)

# ============================================================
# Train Global Classifier (Binary)
# ============================================================
global_clf = xgb.XGBClassifier(
    n_estimators=350,
    max_depth=5,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    verbosity=0
)
global_clf.fit(X_train_global, y_train_bin)

# ============================================================
# Evaluation
# ============================================================
test_pred = global_clf.predict(X_test_global)

print("\nTest Accuracy:", accuracy_score(y_test_bin, test_pred))
print("\nTest Classification Report:")
print(classification_report(
    y_test_bin, test_pred, digits=4,
    target_names=['negative', 'positive']
))

X_train_global shape: (1161, 4)

Test Accuracy: 0.7654639175257731

Test Classification Report:
              precision    recall  f1-score   support

    negative     0.7461    0.9597    0.8395       248
    positive     0.8551    0.4214    0.5646       140

    accuracy                         0.7655       388
   macro avg     0.8006    0.6906    0.7020       388
weighted avg     0.7854    0.7655    0.7403       388



C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
